# Python 101 - Solutions
## Chapter VI

---

**For teaching assistants.** This notebook mirrors the exercises in
`../python101_06.ipynb` one for one. Most solutions end with an `assert`, so
running the whole notebook top to bottom is also a self-test: if it runs clean,
every solution still works.

There is usually more than one right answer - if a student's version passes the
same `assert`, it is correct.

In [ ]:
# Run from the chapter folder, so that `helpers`, `./data/...` and `./pics/...`
# resolve exactly the way they do in the lecture notebooks.
import os
import sys

if os.path.basename(os.getcwd()) == 'solutions':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

print('working directory:', os.getcwd())

> **Note.** This chapter is largely skipped in the current course - objects are introduced at a high level in chapter V instead, just enough to make namespaces and third-party APIs make sense. These solutions are here for completeness.

In [ ]:
import math
import random

from helpers import BouncyBallSimulator, DemoBall, ExampleRPS, RPSApp

### Your turn: the `Student` class

In [ ]:
class Student:
    """A student, with a name and a grade point average."""

    def __init__(self, name, gpa):
        self.name = name
        self.gpa = gpa

    def get_stipend(self, usd_per_gpa):
        """How much this student receives, at `usd_per_gpa` per GPA point."""
        return self.gpa * usd_per_gpa


anna = Student('Anna', 4.5)
print(anna.name, anna.get_stipend(100))

assert anna.get_stipend(100) == 450.0
assert Student('Bob', 0).get_stipend(100) == 0

### Your turn: `SZISZtudent`, inherited from `Student`

`super().__init__(name, gpa)` is the line students most often forget - without it the object has no `name` and no `gpa` at all.

In [ ]:
class SZISZtudent(Student):
    """A SZISZ student: a Student with an extra DSZ score."""

    def __init__(self, name, gpa, dsz_score):
        super().__init__(name, gpa)
        self.dsz_score = dsz_score

    def get_dsz_stipend(self, dsz_base_multiplier):
        return self.dsz_score * dsz_base_multiplier

    def get_final_stipend(self, usd_per_gpa, dsz_base_multiplier):
        return (self.get_stipend(usd_per_gpa)
                + self.get_dsz_stipend(dsz_base_multiplier))


bela = SZISZtudent('Béla', 4.0, 25)
print(bela.name, bela.get_final_stipend(100, 10))

assert bela.get_stipend(100) == 400.0        # inherited
assert bela.get_dsz_stipend(10) == 250
assert bela.get_final_stipend(100, 10) == 650.0
assert isinstance(bela, Student)

### 1. Rock-paper-scissors as a class

`trumps` is a **class** attribute (the rules never change), `hands` is set in the constructor. `helpers.ExampleRPS` is the same thing, so `helpers.test_game` can be used to check your version.

In [ ]:
class RPS:
    """Rock-paper-scissors. `trumps[x]` is the move that beats x."""

    trumps = {'r': 'p', 'p': 's', 's': 'r'}

    def __init__(self):
        self.hands = ['r', 'p', 's']
        self.ai = None

    def move(self):
        return random.choice(self.hands)

    def play(self, hand):
        self.ai = self.move()
        self.hands.append(self.trumps[hand])

        if self.ai == hand:
            return 'draw'
        if self.trumps[self.ai] == hand:
            return 'win'
        return 'lose'


game = RPS()
print(game.play('r'), '| ai played', game.ai)

from helpers import test_game
assert test_game(RPS) is True
assert RPS.trumps['r'] == 'p'

### 2. The cheating version

The trick is that `play` appends `trumps[hand]` to `self.hands` - the move that *would have beaten* the player. Since `move()` picks randomly from `hands`, the AI's picks drift towards whatever counters the player's habits. The longer you play, the more it cheats.

Because `RPS.play` already does the appending, the cheater only needs to override `move` and remember the history.

In [ ]:
class CheatingRPS(RPS):
    """Weights its own moves by what would have beaten you so far."""

    def __init__(self):
        super().__init__()
        self.history = []

    def move(self):
        # `hands` grows with the counter to every move the player made
        return random.choice(self.hands)

    def play(self, hand):
        self.history.append(hand)
        return super().play(hand)


cheater = CheatingRPS()
for _ in range(30):
    cheater.play('r')          # a very predictable player

print('history length :', len(cheater.history))
print('hands now      :', len(cheater.hands), 'entries')
print("share of 'p'   :", round(cheater.hands.count('p') / len(cheater.hands), 2))

assert test_game(CheatingRPS) is True
assert len(cheater.history) == 30
# 'p' beats 'r', so the bag should be dominated by 'p' by now
assert cheater.hands.count('p') > cheater.hands.count('s')

results = [CheatingRPS().play('r') for _ in range(1)]  # smoke test
assert results[0] in ('win', 'lose', 'draw')

### Extra - GUI

`helpers.RPSApp` builds a tkinter window and validates your class first. It blocks the notebook while the window is open.

```python
RPSApp(CheatingRPS).run()
```

Not run here, because it would hang this notebook - but `test_game` above already checked that `RPSApp` would accept the class.

### 3. The one ring

In [ ]:
class OneRing:
    """The One Ring. There is only one real owner."""

    real_owner = 'Sauron'

    def __init__(self, bearer, description='One Ring to rule them all'):
        self.bearer = bearer
        self.description = description
        self.worn = False

    def wear(self):
        self.worn = True
        print(f"Hi {self.real_owner}! I'm {self.bearer}, "
              f"and I'm here, wearing your ring!")

    def take_off(self):
        self.worn = False

    def turn_invisible(self):
        """The ring's special power - only works while it is worn."""
        if not self.worn:
            return f'{self.bearer} is perfectly visible.'
        return f'{self.bearer} vanishes, and Sauron feels a chill...'


ring = OneRing('Frodo')
print(ring.turn_invisible())
ring.wear()
print(ring.turn_invisible())

assert OneRing.real_owner == 'Sauron'
assert not OneRing('Bilbo').worn
assert ring.worn and 'vanishes' in ring.turn_invisible()
ring.take_off()
assert 'visible' in ring.turn_invisible()

### 4. Geometry: `Point` and `Line`

Note the notebook calls the class `Point` here but refers to `Point2d` in exercise 6 - an inconsistency in the text. `Point` is used throughout below.

In [ ]:
class Point:
    """A point in 2D space."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Point({self.x}, {self.y})'


class Line:
    """A line between two Points."""

    def __init__(self, start, end):
        self.start = start
        self.end = end

    def length(self):
        """Euclidean distance between the two endpoints."""
        return math.sqrt((self.start.x - self.end.x) ** 2
                         + (self.start.y - self.end.y) ** 2)


A = Point(0, 0)
B = Point(1, 1)
v = Line(A, B)
print(A, B, round(v.length(), 2))

assert abs(v.length() - 1.41) < 0.01
assert Line(Point(0, 0), Point(3, 4)).length() == 5.0
assert Line(A, A).length() == 0.0

### 5. `StudentInsight`

The `metric` parameter is a good excuse to show `statistics` from the standard library rather than writing a standard deviation by hand.

In [ ]:
import statistics


class StudentInsight:
    """Collects Students and reports on their GPAs and stipends."""

    def __init__(self, usd_per_gpa):
        self.usd_per_gpa = usd_per_gpa
        self.students = []

    def add(self, student):
        self.students.append(student)
        return self

    def _values(self, target):
        if target == 'gpa':
            return [student.gpa for student in self.students]
        if target == 'stipend':
            return [student.get_stipend(self.usd_per_gpa)
                    for student in self.students]
        raise ValueError(f"target must be 'gpa' or 'stipend', got {target!r}")

    def calculate(self, target='gpa', metric='avg'):
        values = self._values(target)
        if metric == 'avg':
            return statistics.fmean(values)
        if metric == 'sum':
            return sum(values)
        if metric == 'std':
            return statistics.stdev(values)
        raise ValueError(f"metric must be 'avg', 'sum' or 'std', got {metric!r}")


insight = StudentInsight(usd_per_gpa=100)
for name, gpa in [('Anna', 4.5), ('Béla', 4.0), ('Cecil', 3.0), ('Dóra', 5.0)]:
    insight.add(Student(name, gpa))

print('avg gpa      :', insight.calculate())
print('sum stipend  :', insight.calculate(target='stipend', metric='sum'))
print('std of gpas  :', round(insight.calculate(metric='std'), 3))

assert insight.calculate(target='gpa', metric='sum') == 16.5
assert insight.calculate(target='stipend', metric='sum') == 1650.0
assert insight.calculate(target='gpa', metric='avg') == 4.125
try:
    insight.calculate(target='height')
    raise AssertionError('should have rejected an unknown target')
except ValueError as error:
    print('\nvalidation:', error)

### 5.b Extra: it already works with `SZISZtudent`s

Because `SZISZtudent` **is a** `Student`, it has `gpa` and `get_stipend` - so `StudentInsight` needs no changes at all. That is the point of inheritance, and it is worth making explicit.

In [ ]:
mixed = StudentInsight(usd_per_gpa=100)
mixed.add(Student('Anna', 4.5))
mixed.add(SZISZtudent('Béla', 4.0, 25))

print('sum stipend:', mixed.calculate(target='stipend', metric='sum'))

# the plain Student stipend is used for both - the DSZ part needs a subclass
# that knows about it, which is the real extra exercise:
class SZISZInsight(StudentInsight):
    def __init__(self, usd_per_gpa, dsz_base_multiplier):
        super().__init__(usd_per_gpa)
        self.dsz_base_multiplier = dsz_base_multiplier

    def _values(self, target):
        if target == 'stipend':
            return [
                student.get_final_stipend(self.usd_per_gpa, self.dsz_base_multiplier)
                if isinstance(student, SZISZtudent)
                else student.get_stipend(self.usd_per_gpa)
                for student in self.students
            ]
        return super()._values(target)


szisz = SZISZInsight(usd_per_gpa=100, dsz_base_multiplier=10)
szisz.add(Student('Anna', 4.5))
szisz.add(SZISZtudent('Béla', 4.0, 25))
print('sum stipend with DSZ:', szisz.calculate(target='stipend', metric='sum'))

assert mixed.calculate(target='stipend', metric='sum') == 850.0
assert szisz.calculate(target='stipend', metric='sum') == 1100.0

### 6. CHALLENGE: the bouncy ball

Inherits position from `Point`. The bounce logic: work out where the ball *would* land, and if that is outside the box, flip the velocity on that axis and recompute.

In [ ]:
class Ball(Point):
    """A ball that bounces around inside a box."""

    def __init__(self, x, y, vx=1, vy=1, max_x=5, max_y=7):
        super().__init__(x, y)
        self.vx = vx
        self.vy = vy
        self.max_x = max_x
        self.max_y = max_y

    def step(self):
        next_x = self.x + self.vx
        next_y = self.y + self.vy

        if next_x >= self.max_x or next_x < 0:
            self.vx *= -1
            next_x = self.x + self.vx

        if next_y >= self.max_y or next_y < 0:
            self.vy *= -1
            next_y = self.y + self.vy

        self.x = next_x
        self.y = next_y


ball = Ball(x=0, y=0, vx=1, vy=1, max_x=5, max_y=40)
positions = []
for _ in range(200):
    ball.step()
    positions.append((ball.x, ball.y))

assert all(0 <= x < 5 and 0 <= y < 40 for x, y in positions), 'escaped the box!'
assert isinstance(ball, Point)
# it must actually bounce, not sit still
assert len(set(positions)) > 10
print(f'200 steps, stayed inside, visited {len(set(positions))} distinct cells')
print(positions[:10])

And in the widget (run this cell, then press *start*):

In [ ]:
BouncyBallSimulator(Ball(x=0, y=0, vx=1, vy=1, max_x=5, max_y=40)).show()